<a href="https://colab.research.google.com/github/rajmishra-dev/ML-Learning-project/blob/main/ab-test/ml_ad_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing a Libray.!

In [1]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
random.seed(55)

In [2]:
df = pd.read_csv('/content/ab_data.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19417 entries, 0 to 19416
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       19417 non-null  int64  
 1   timestamp     19417 non-null  object 
 2   group         19417 non-null  object 
 3   landing_page  19417 non-null  object 
 4   converted     19416 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 758.6+ KB


In [4]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0.0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0.0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0.0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0.0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1.0


In [5]:
df.groupby(['group','converted']).agg('count')

user_id  timestamp  landing_page
group     converted                                  
control   0.0           8523       8523          8523
          1.0           1176       1176          1176
treatment 0.0           8508       8508          8508
          1.0           1209       1209          1209

In [6]:
df.drop(df.query("group == 'control' and landing_page == 'new_page'").index, inplace=True)
df.drop(df.query("group == 'treatment' and landing_page == 'old_page'").index, inplace=True)

df.groupby(['group','converted']).agg('count')

user_id  timestamp  landing_page
group     converted                                  
control   0.0           8402       8402          8402
          1.0           1158       1158          1158
treatment 0.0           8399       8399          8399
          1.0           1187       1187          1187

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19147 entries, 0 to 19416
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       19147 non-null  int64  
 1   timestamp     19147 non-null  object 
 2   group         19147 non-null  object 
 3   landing_page  19147 non-null  object 
 4   converted     19146 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 897.5+ KB


###Cleaning Duplicated Users

In [8]:
df[df.duplicated(['user_id'], keep=False)]


,user_id,timestamp,group,landing_page,converted
1899,773192,2017-01-09 05:37:58.781806,treatment,new_page,0.0
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0.0


In [9]:
df.drop_duplicates(['user_id'], inplace=True)

In [10]:
assert len(df['user_id'].unique()) == df['user_id'].size

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19146 entries, 0 to 19416
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       19146 non-null  int64  
 1   timestamp     19146 non-null  object 
 2   group         19146 non-null  object 
 3   landing_page  19146 non-null  object 
 4   converted     19145 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 897.5+ KB


In [12]:
df['converted'].mean()

np.float64(0.12248628884826325)

In [13]:
df.groupby(['group']).describe()

user_id                                                             \
            count           mean           std       min       25%       50%   
group                                                                          
control    9560.0  787252.241213  91107.430746  630025.0  708548.5  786994.0   
treatment  9586.0  787977.294075  91055.666966  630042.0  709217.5  787291.5   

                               converted                                     \
                 75%       max     count      mean       std  min  25%  50%   
group                                                                         
control    866019.25  945985.0    9560.0  0.121130  0.326295  0.0  0.0  0.0   
treatment  866008.25  945940.0    9585.0  0.123839  0.329415  0.0  0.0  0.0   

                     
           75%  max  
group                
control    0.0  1.0  
treatment  0.0  1.0

In [14]:
df.groupby(['group']).agg({'converted' : ['sum', 'count', 'mean']})

converted                
                sum count      mean
group                              
control      1158.0  9560  0.121130
treatment    1187.0  9585  0.123839

In [15]:
df.groupby(['group']).agg({'converted' : ['sum', 'count', 'mean']})

converted                
                sum count      mean
group                              
control      1158.0  9560  0.121130
treatment    1187.0  9585  0.123839

In [16]:
df[['group','converted']].groupby(['group']).agg('mean').T

group,control,treatment
converted,0.12113,0.123839


Calculate Prob. of New and Old Pages
Respectively

In [17]:
p_old_page = df[['group', 'converted']].query("group == 'control'")['converted'].mean()
p_new_page = df[['group', 'converted']].query("group == 'treatment'")['converted'].mean()
act_p_diff = p_new_page - p_old_page

print('p_old_page:\t{}\np_new_page:\t{}\np_diff:\t\t{}'.format(p_old_page, p_new_page, act_p_diff))

p_old_page:	0.1211297071129707
p_new_page:	0.12383933229003652
p_diff:		0.0027096251770658136


###Calculate Count of New and Old Pages Respectively

In [18]:
n_old = len(df[['group']].query("group == 'control'"))


In [19]:
n_new = len(df[['group']].query("group == 'treatment'"))